<a href="https://colab.research.google.com/github/naimul-islam-64/Backup/blob/main/Tool_Calling_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 📦 Cell 1 - Install dependencies
# Cell 1: Install all required packages
!pip install unsloth --quiet
!pip install trl datasets transformers accelerate bitsandbytes peft --quiet
!pip install torch --quiet

print("✅ All packages installed!")

In [ ]:
# @title 🔍 Cell 2 — Imports & Config

# Cell 2: Imports and Configuration
import torch
import json
from datasets import load_dataset
from transformers import TrainingArguments, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel

# ─── CONFIG ───────────────────────────────────────────────
MODEL_NAME       = "ibm-granite/granite-4.0-350m"
MAX_SEQ_LENGTH   = 2048
LORA_RANK        = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
BATCH_SIZE       = 4
GRAD_ACCUM       = 4
LEARNING_RATE    = 2e-4
NUM_EPOCHS       = 3
WARMUP_RATIO     = 0.05
OUTPUT_DIR       = "./granite-tool-calling"
DATASET_NAME     = "Salesforce/xlam-function-calling-60k"
DATASET_SPLIT    = "train"
MAX_SAMPLES      = 20000   # Reduce if running out of memory; full = 60k
# ──────────────────────────────────────────────────────────

print(f"✅ Config loaded | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"   VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# @title 🤖 Cell— 3  Load Model & Apply LoRA

# Cell 3: Load Base Model and Apply LoRA with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,       # Auto: bf16 on Ampere+, fp16 otherwise
    load_in_4bit    = True,       # QLoRA — saves VRAM
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_RANK,
    target_modules      = ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",   # Saves memory
    random_state        = 42,
)

# Print trainable params summary
model.print_trainable_parameters()
print("✅ Model loaded and LoRA applied!")

In [ ]:
# @title 📊 Cell —4  Load & Inspect Dataset

# Cell 4: Load and Inspect the Dataset
raw_dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)

# Subsample if needed
if MAX_SAMPLES and MAX_SAMPLES < len(raw_dataset):
    raw_dataset = raw_dataset.shuffle(seed=42).select(range(MAX_SAMPLES))

print(f"✅ Dataset loaded: {len(raw_dataset)} samples")
print(f"\n📋 Column names: {raw_dataset.column_names}")
print(f"\n🔍 Sample entry:")
sample = raw_dataset[0]
for k, v in sample.items():
    print(f"\n  [{k}]:\n  {str(v)[:300]}...")

In [ ]:
# @title 🔧 Cell— 5  Format Dataset for Tool Calling

# Cell 5: Format Dataset into Tool Calling Chat Template

SYSTEM_PROMPT = """You are a helpful AI assistant with access to tools/functions.
When the user's request requires a tool, respond ONLY with a JSON tool call in this exact format:
{"name": "function_name", "arguments": {"arg1": "value1", "arg2": "value2"}}
If no tool is needed, respond naturally in plain text.
Always be accurate and concise."""

def format_xlam_sample(sample):
    """
    xlam-function-calling-60k schema:
      - tools: JSON string of available tool definitions
      - query: user question
      - answers: JSON string of expected tool calls
    """
    try:
        tools    = json.loads(sample["tools"])   if isinstance(sample["tools"], str)   else sample["tools"]
        answers  = json.loads(sample["answers"]) if isinstance(sample["answers"], str) else sample["answers"]
        query    = sample["query"]
    except (json.JSONDecodeError, KeyError):
        return None

    # Build tool descriptions for system prompt
    tool_descriptions = []
    for tool in tools:
        name = tool.get("name", "")
        desc = tool.get("description", "")
        params = tool.get("parameters", {})
        props = params.get("properties", {})
        required = params.get("required", [])

        param_strs = []
        for pname, pinfo in props.items():
            req_marker = "*" if pname in required else ""
            param_strs.append(f"  - {pname}{req_marker}: {pinfo.get('description', '')} ({pinfo.get('type', 'any')})")

        tool_descriptions.append(
            f"Tool: {name}\nDescription: {desc}\nParameters:\n" + "\n".join(param_strs)
        )

    tools_block = "\n\n".join(tool_descriptions)

    # Build the expected assistant response (tool call as JSON)
    if isinstance(answers, list) and len(answers) > 0:
        answer = answers[0]
    else:
        answer = answers

    assistant_response = json.dumps(answer, ensure_ascii=False)

    # Final chat-formatted prompt
    full_prompt = f"""<|system|>
{SYSTEM_PROMPT}

Available Tools:
{tools_block}
<|user|>
{query}
<|assistant|>
{assistant_response}"""

    return {"text": full_prompt}


# Apply formatting
formatted_dataset = raw_dataset.map(
    format_xlam_sample,
    remove_columns=raw_dataset.column_names,
    desc="Formatting dataset"
)

# Remove any None entries (malformed samples)
formatted_dataset = formatted_dataset.filter(lambda x: x["text"] is not None)

print(f"✅ Dataset formatted: {len(formatted_dataset)} samples")
print(f"\n📝 Sample formatted prompt:\n")
print(formatted_dataset[0]["text"][:800] + "\n...")

In [ ]:
# @title ⚙️ Cell 6 — Training Arguments & Trainer Setup

# Cell 6: Configure Training Arguments and SFTTrainer

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_ratio                = WARMUP_RATIO,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    optim                       = "adamw_8bit",   # Memory efficient
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 50,
    save_strategy               = "steps",
    save_steps                  = 500,
    save_total_limit            = 2,
    seed                        = 42,
    report_to                   = "none",         # Change to "wandb" if you want logging
    dataset_text_field          = "text",
    max_seq_length              = MAX_SEQ_LENGTH,
    packing                     = True,            # Packs short sequences → faster training
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = formatted_dataset,
    args            = training_args,
)

print("✅ Trainer initialized!")
print(f"   Total training steps: {trainer.args.max_steps if trainer.args.max_steps > 0 else 'auto (epoch-based)'}")
print(f"   Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# @title 🚀 Cell 7 — Train!

# Cell 7: Start Training
print("🚀 Starting training...\n")

# Show GPU memory before training
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   GPU Memory before training: {used:.2f} / {total:.2f} GB\n")

trainer_stats = trainer.train()

# Training summary
print("\n✅ Training complete!")
print(f"   Training loss    : {trainer_stats.training_loss:.4f}")
print(f"   Total time       : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"   Samples/sec      : {trainer_stats.metrics['train_samples_per_second']:.2f}")

In [ ]:
# @title 💾 Cell 8 — Save Model (LoRA Adapter + Optional Merge)

# Cell 8: Save the fine-tuned model

ADAPTER_SAVE_PATH = "./granite-tool-lora-adapter"
MERGED_SAVE_PATH  = "./granite-tool-merged"

# 1. Save just the LoRA adapter (lightweight, ~20-50MB)
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f"✅ LoRA adapter saved → {ADAPTER_SAVE_PATH}")

# 2. Optional: Save merged (full) model in 16-bit
# (Comment out if you're low on disk/RAM in Colab)
model.save_pretrained_merged(
    MERGED_SAVE_PATH,
    tokenizer,
    save_method="merged_16bit"
)
print(f"✅ Merged 16-bit model saved → {MERGED_SAVE_PATH}")

# 3. Optional: Export to GGUF for llama.cpp / Ollama deployment
# model.save_pretrained_gguf("granite-tool-gguf", tokenizer, quantization_method="q4_k_m")
# print("✅ GGUF model exported!")

In [ ]:
# @title 🧪 Cell— 9  Inference Test

# Cell 9: Test the fine-tuned model with tool calling

FastLanguageModel.for_inference(model)  # Enable optimized inference mode

def run_tool_call_inference(user_query, tools_list):
    tool_descriptions = []
    for tool in tools_list:
        props = tool.get("parameters", {}).get("properties", {})
        required = tool.get("parameters", {}).get("required", [])
        param_strs = [
            f"  - {p}{' *' if p in required else ''}: {info.get('description','')} ({info.get('type','any')})"
            for p, info in props.items()
        ]
        tool_descriptions.append(
            f"Tool: {tool['name']}\nDescription: {tool['description']}\nParameters:\n" + "\n".join(param_strs)
        )

    prompt = f"""<|system|>
{SYSTEM_PROMPT}

Available Tools:
{"\\n\\n".join(tool_descriptions)}
<|user|>
{user_query}
<|assistant|>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens  = 256,
            temperature     = 0.1,
            do_sample       = True,
            pad_token_id    = tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()


# ── Test 1: Weather tool ──────────────────────────────────
weather_tools = [{
    "name": "get_current_weather",
    "description": "Get the current weather for a given city",
    "parameters": {
        "properties": {
            "city":  {"description": "City name", "type": "string"},
            "units": {"description": "Temperature units: celsius or fahrenheit", "type": "string"}
        },
        "required": ["city"]
    }
}]

print("=" * 60)
print("🧪 Test 1: Weather query")
print("User: What's the weather like in Tokyo right now?")
response1 = run_tool_call_inference("What's the weather like in Tokyo right now?", weather_tools)
print(f"Model: {response1}")

# ── Test 2: Search tool ───────────────────────────────────
search_tools = [{
    "name": "web_search",
    "description": "Search the internet for current information",
    "parameters": {
        "properties": {
            "query":       {"description": "Search query string", "type": "string"},
            "num_results": {"description": "Number of results to return", "type": "integer"}
        },
        "required": ["query"]
    }
}]

print("\n" + "=" * 60)
print("🧪 Test 2: Search query")
print("User: Find the latest news about AI in 2025")
response2 = run_tool_call_inference("Find the latest news about AI in 2025", search_tools)
print(f"Model: {response2}")

# ── Test 3: No tool needed ────────────────────────────────
print("\n" + "=" * 60)
print("🧪 Test 3: No tool needed (plain response expected)")
print("User: What is 2 + 2?")
response3 = run_tool_call_inference("What is 2 + 2?", weather_tools)
print(f"Model: {response3}")
print("=" * 60)